In [33]:
%pip install yfinance pandas numpy matplotlib

Note: you may need to restart the kernel to use updated packages.


In [35]:
%pip install jinja2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [jinja2]
Note: you may need to restart the kernel to use updated packages.


In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

Matplotlib is building the font cache; this may take a moment.


In [6]:
TICKERS = ["SPY", "IEF", "GLD", "DBC"]

START_DATE = "2007-01-01"
END_DATE = "2026-01-01" # end date is exclusive, so our requests of data are through to the end of 2025.

### OHLC PRICES

Refers to: Open, High, Low, Close prices.

open = when the trading session begins
high = highest price during session
low = lowest
close = when trading session ends

#### ok but what is auto-adjust?

basically the issue is when a stock's price changes because of corporate actions, like dividend payouts.

for example, when a company pays a $2 dividend per share, the stock's worth may drop by $2. `auto_adjust` gives the unadjusted OHLC prices, saying that the calculated return is -2%.

In [7]:
raw = yf.download(
    tickers=TICKERS,
    start=START_DATE,
    end=END_DATE,
    interval="1d", # we're retrieving one row of data per trading day, can be 1wk, 1mo, 1h etc.
    auto_adjust=False, # this means we are gathering unadjusted OHLC prices.
    actions=True, # inlcude corporate actions like dividends or stock splits.
    group_by="column",
)

if raw is None or raw.empty:
    raise RuntimeError("no data downloaded!")

raw.head()

[*********************100%***********************]  4 of 4 completed


Price       Adj Close                                  Capital Gains       \
Ticker            DBC        GLD        IEF        SPY           DBC  GLD   
Date                                                                        
2007-01-03  19.330030  62.279999  49.789158  98.624603           0.0  0.0   
2007-01-04  18.826475  61.650002  49.969776  98.833878           0.0  0.0   
2007-01-05  18.761503  60.169998  49.807159  98.045563           0.0  0.0   
2007-01-08  18.858961  60.480000  49.843327  98.499008           0.0  0.0   
2007-01-09  18.696526  60.849998  49.831295  98.415314           0.0  0.0   

Price                     Close             ...       Open              \
Ticker      IEF  SPY        DBC        GLD  ...        IEF         SPY   
Date                                        ...                          
2007-01-03  0.0  0.0  23.799999  62.279999  ...  82.709999  142.250000   
2007-01-04  0.0  0.0  23.180000  61.650002  ...  82.830002  141.229996   
2007-01-05  0.0  0.0  23.100000  60.169998  ...  82.580002  141.330002   
2007-01-08  0.0  0.0  23.219999  60.480000  ...  82.720001  140.820007   
2007-01-09  0.0  0.0  23.020000  60.849998  ...  82.820000  141.309998   

Price      Stock Splits                 Volume                               
Ticker              DBC  GLD  IEF  SPY     DBC       GLD      IEF       SPY  
Date                                                                         
2007-01-03          0.0  0.0  0.0  0.0  746800   8061900  1210700  94807600  
2007-01-04          0.0  0.0  0.0  0.0  428900   4858600   171200  69620600  
2007-01-05          0.0  0.0  0.0  0.0  369800  12500100   201000  76645300  
2007-01-08          0.0  0.0  0.0  0.0  275200   3478700   163800  71655000  
2007-01-09          0.0  0.0  0.0  0.0  675600   4170400   168600  75680100  

[5 rows x 36 columns]

In [ ]:
prices = (
    raw["Adj Close"]
    .reindex(columns=TICKERS) # helps order based on my TICKERS list
    # .sort_index(ascending=False) -> shows the latest dates
    .sort_index() 
)

prices.head()

Ticker,SPY,IEF,GLD,DBC
Date,,,,
2007-01-03,98.624603,49.789158,62.279999,19.330030
2007-01-04,98.833878,49.969776,61.650002,18.826475
2007-01-05,98.045563,49.807159,60.169998,18.761503
2007-01-08,98.499008,49.843327,60.480000,18.858961
2007-01-09,98.415314,49.831295,60.849998,18.696526


In [18]:
coverage = pd.DataFrame({
    "first_date": prices.apply(lambda column: column.first_valid_index()),
    "last_date": prices.apply(lambda column: column.last_valid_index()),
    "observations": prices.notna().sum(),
    "missing_values": prices.isna().sum(),
})

display(coverage)

print("Duplicate dates:", prices.index.duplicated().sum())
print("Non-positive prices:", prices.le(0).sum().to_dict())

,first_date,last_date,observations,missing_values
Ticker,,,,
SPY,2007-01-03,2025-12-31,4780,0
IEF,2007-01-03,2025-12-31,4780,0
GLD,2007-01-03,2025-12-31,4780,0
DBC,2007-01-03,2025-12-31,4780,0


Duplicate dates: 0
Non-positive prices: {'SPY': 0, 'IEF': 0, 'GLD': 0, 'DBC': 0}


In [40]:
daily_returns = prices.pct_change(fill_method=None)

daily_returns.head() # first row should be all NaN because there is no preceding price in the DS.

Ticker,SPY,IEF,GLD,DBC
Date,,,,
2007-01-03,NaN,NaN,NaN,NaN
2007-01-04,0.002122,0.003628,-0.010116,-0.026050
2007-01-05,-0.007976,-0.003254,-0.024007,-0.003451
2007-01-08,0.004625,0.000726,0.005152,0.005195
2007-01-09,-0.000850,-0.000241,0.006118,-0.008613


In [41]:
daily_returns.tail()

Ticker,SPY,IEF,GLD,DBC
Date,,,,
2025-12-24,0.003518,0.002601,-0.004134,-0.000442
2025-12-26,-0.000101,0.000934,0.011677,0.003093
2025-12-29,-0.003564,0.001452,-0.043528,-0.009251
2025-12-30,-0.001221,-0.001035,0.000728,0.006670
2025-12-31,-0.007409,-0.003317,-0.006468,-0.012367


In [42]:
daily_returns_pct = daily_returns * 100
daily_returns_pct.head().style.format("{:.2%}")
# daily_returns_pct.tail().style.format("{:.2%}")

Ticker,SPY,IEF,GLD,DBC
Date,,,,
2007-01-03 00:00:00,nan%,nan%,nan%,nan%
2007-01-04 00:00:00,21.22%,36.28%,-101.16%,-260.50%
2007-01-05 00:00:00,-79.76%,-32.54%,-240.07%,-34.51%
2007-01-08 00:00:00,46.25%,7.26%,51.52%,51.95%
2007-01-09 00:00:00,-8.50%,-2.41%,61.18%,-86.13%


In [44]:
daily_returns = daily_returns.iloc[1:].copy()
assert daily_returns.notna().all().all(), "Unexpected missing returns"
assert np.isfinite(daily_returns.to_numpy()).all(), "Infinite returns found"

print("Return observations per ETF:")
display(daily_returns.count())

Return observations per ETF:


Ticker
SPY    4779
IEF    4779
GLD    4779
DBC    4779
dtype: int64

In [46]:
spy_example = pd.DataFrame({
    "previous_adjusted_close": prices["SPY"].shift(1),
    "current_adjusted_close": prices["SPY"],
    "daily_return_pdct": daily_returns["SPY"] * 100,
})

In [50]:
spy_example.iloc[1:10]

,previous_adjusted_close,current_adjusted_close,daily_return_pdct
Date,,,
2007-01-04,98.624603,98.833878,0.212193
2007-01-05,98.833878,98.045563,-0.797616
2007-01-08,98.045563,98.499008,0.462484
2007-01-09,98.499008,98.415314,-0.084970
2007-01-10,98.415314,98.743195,0.333160
2007-01-11,98.743195,99.175713,0.438023
2007-01-12,99.175713,99.929207,0.759757
2007-01-16,99.929207,99.733856,-0.195489
2007-01-17,99.733856,99.775711,0.041967


In [ ]:
summary = daily_returns.agg(["mean", "std", "min", "max"]) * 100
# gives us the average daily return, standard deviation of daily returns, worst and best daily returns.
display(summary.round(3))

Ticker,SPY,IEF,GLD,DBC
mean,0.048,0.014,0.045,0.010
std,1.245,0.440,1.103,1.207
min,-10.942,-2.507,-8.781,-7.944
max,14.520,3.426,11.291,6.874


From this aggregated table / summary table, we can see that IEF had the **lowest daily volatility** of 0.440%, while SPY had the **highest daily volatility** at 1.245%. 

Now, imagine we are at 30 June 2015, and we are trying to forecast July 2015's volatility. We will use April, May and June (3-month lookback) in order to forecast July's volatility.

In [56]:
lookback_returns = daily_returns.loc[
    "2015-04-01" : "2015-06-30"
]

forecast = lookback_returns.std(ddof=1) # calculates the sliced standard dev.

display(
    (forecast * 100)
    .rename("forecast_daily_volatility_percentage")
    .to_frame()
)


,forecast_daily_volatility_percentage
Ticker,
SPY,0.635478
IEF,0.436554
GLD,0.809238
DBC,0.948290


Attached are the forecasted volatities based on a 3-month lookback.

Next, we will display the realised volatility.

In [55]:
next_month_returns = daily_returns.loc[
    "2015-07-01":"2015-07-31"
]

actual = next_month_returns.std(ddof=1)

display(
    (actual * 100)
    .rename("actual_daily_volatility_pct")
    .to_frame()
)

,actual_daily_volatility_pct
Ticker,
SPY,0.733815
IEF,0.389163
GLD,0.744540
DBC,0.998825


Now, compare the forecast and the realised outcomes:

In [58]:
comparison = pd.DataFrame({
    "forecast_pct": forecast * 100,
    "actual_pct": actual * 100,
    "error_pp": (forecast - actual) * 100,
    "absolute_error_pp": (forecast - actual).abs() * 100,
})

display(comparison.round(3))

,forecast_pct,actual_pct,error_pp,absolute_error_pp
Ticker,,,,
SPY,0.635,0.734,-0.098,0.098
IEF,0.437,0.389,0.047,0.047
GLD,0.809,0.745,0.065,0.065
DBC,0.948,0.999,-0.051,0.051


In [59]:
comparison["absolute_relative_error_pct"] = (
    (forecast - actual).abs() / actual
) * 100

display(comparison.round(3))

,forecast_pct,actual_pct,error_pp,absolute_error_pp,absolute_relative_error_pct
Ticker,,,,,
SPY,0.635,0.734,-0.098,0.098,13.401
IEF,0.437,0.389,0.047,0.047,12.178
GLD,0.809,0.745,0.065,0.065,8.690
DBC,0.948,0.999,-0.051,0.051,5.059


Here, we can view the relative errors in percentages.

In [60]:
window_starts = {
    "3_month": "2015-04-01",
    "6_month": "2015-01-01",
    "12_month": "2014-07-01",
}

forecast_date = "2015-06-30"

forecasts = pd.DataFrame({
    name: daily_returns.loc[start:forecast_date].std(ddof=1)
    for name, start in window_starts.items()
})

display((forecasts * 100).round(3))

,3_month,6_month,12_month
Ticker,,,
SPY,0.635,0.762,0.755
IEF,0.437,0.464,0.395
GLD,0.809,0.912,0.961
DBC,0.948,1.092,0.982


In [61]:
actual = daily_returns.loc[
    "2015-07-01":"2015-07-31"
].std(ddof=1)

forecast_comparison = forecasts.copy()
forecast_comparison["actual"] = actual

display((forecast_comparison * 100).round(3))

,3_month,6_month,12_month,actual
Ticker,,,,
SPY,0.635,0.762,0.755,0.734
IEF,0.437,0.464,0.395,0.389
GLD,0.809,0.912,0.961,0.745
DBC,0.948,1.092,0.982,0.999


In [62]:
absolute_errors = forecasts.sub(actual, axis="index").abs()

# Express errors in percentage points.
error_comparison = absolute_errors * 100

error_comparison["best_this_month"] = absolute_errors.idxmin(axis=1)

display(error_comparison.round(3))

,3_month,6_month,12_month,best_this_month
Ticker,,,,
SPY,0.098,0.029,0.021,12_month
IEF,0.047,0.074,0.006,12_month
GLD,0.065,0.167,0.216,3_month
DBC,0.051,0.093,0.017,12_month


#### A conclusion we can make:

For forecasting July 2015's volatility across three out of the four ETFs, the 12-month estimate produced the smallest absolute relative error. The larger GLD mistake offsets the 12-month method advantages elsewhere.

Further evaluation across time is required. We have not considered additional information to determine whether they may improve forecasts beyond our volatility-only baseline.